<img src="https://www.unad.edu.co/images/footer/logo-unad-acreditacion-min.png" width="780" height="140" align="right"/>

<p style="text-align: center;"> Curso: ENSEMBLE METHODS AND KERNELS</p>

<p style="text-align: center;"> Código Curso: 203008076 </p>

<p style="text-align: center;"> Grupo: 1 </p>

<p style="text-align: center;"> Phase 3 -Development of the Practical Component of the
Course Ensemble Methods and Kernels</p>

<p style="text-align: center;">  Presentado por: Wilmer Ricardo Urda</p>

<p style="text-align: center;"> Código: 1017194627</p>

<p style="text-align: center;">  Tutor: Ing. Jorge Luis Quintero Lopez </p>

<p style="text-align: center;"> UNIVERSIDAD NACIONAL ABIERTA Y A DISTANCIA - UNAD </p>

EJERCICIO 2 – BOOSTING

##BOOSTING – REGRESIÓN

In [ ]:
def plot_learning_curve(model, X, y, preprocessor, title, scoring='r2'):
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    train_sizes, train_scores, test_scores = learning_curve(
        pipeline, X, y,
        cv=5,
        scoring=scoring,
        train_sizes=np.linspace(0.1, 1.0, 8),
        n_jobs=-1
    )

    train_mean = train_scores.mean(axis=1)
    test_mean  = test_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    test_std   = test_scores.std(axis=1)

    label = 'R2' if scoring == 'r2' else scoring.capitalize()

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, marker='o', label=f'Train {label}')
    plt.plot(train_sizes, test_mean,  marker='o', label=f'Test {label}')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)
    plt.fill_between(train_sizes, test_mean  - test_std,  test_mean  + test_std,  alpha=0.15)
    plt.title(title)
    plt.xlabel('Training Size')
    plt.ylabel(label)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error

import xgboost as xgb

# =========================
# CARGA DE DATOS
# =========================
data = fetch_openml(data_id=566, as_frame=True)
df = data.frame.copy()

target_col = data.target_names[0]
X = df.drop(columns=[target_col])
y = pd.to_numeric(df[target_col], errors='coerce')

mask = y.notna()
X = X.loc[mask]
y = y.loc[mask]

# =========================
# IDENTIFICAR TIPOS DE COLUMNAS
# =========================
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print('Columnas categoricas:', cat_cols)
print('Columnas numericas:', num_cols)

# =========================
# PREPROCESAMIENTO
# =========================
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

# =========================
# SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# MODELOS
# =========================
models = {
    'AdaBoost': AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=3),
        n_estimators=200,
        learning_rate=0.5,
        random_state=42
    ),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        random_state=42
    ),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=100,
        random_state=42,
        objective='reg:squarederror'
    )
}

# =========================
# ENTRENAMIENTO Y METRICAS
# =========================
results_reg = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results_reg.append({
        'Model':  name,
        'R2':     round(r2_score(y_test, y_pred), 4),
        'RMSE':   round(np.sqrt(mean_squared_error(y_test, y_pred)), 4)
    })

results_reg_df = pd.DataFrame(results_reg)
print(results_reg_df.to_string(index=False))


##Learning curves para regresión

In [ ]:
for name, model in models.items():
    plot_learning_curve(model, X, y, preprocessor,
                        f'Learning Curve - {name} (Regression)')


##BOOSTING – CLASIFICACIÓN

In [ ]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
import xgboost as xgb

# =========================
# CARGA DE DATOS (kr-vs-kp - ID 3)
# =========================
data_clf = fetch_openml(data_id=3, as_frame=True)
df_clf = data_clf.frame.copy()

target_col_clf = data_clf.target_names[0]
X_clf = df_clf.drop(columns=[target_col_clf])
y_clf = LabelEncoder().fit_transform(df_clf[target_col_clf])

# =========================
# PREPROCESAMIENTO
# =========================
cat_cols_clf = X_clf.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols_clf = X_clf.select_dtypes(include=[np.number]).columns.tolist()

numeric_transformer_clf = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer_clf = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_clf = ColumnTransformer(transformers=[
    ('num', numeric_transformer_clf, num_cols_clf),
    ('cat', categorical_transformer_clf, cat_cols_clf)
])

# =========================
# SPLIT (con stratify)
# =========================
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# =========================
# MODELOS
# =========================
models_clf = {
    'AdaBoost': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
        n_estimators=100,
        learning_rate=0.5,
        random_state=42
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        random_state=42
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    )
}

# =========================
# EVALUACION
# =========================
results_clf = []

for name, model in models_clf.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor_clf),
        ('model', model)
    ])

    pipeline.fit(X_train_clf, y_train_clf)
    y_pred_clf = pipeline.predict(X_test_clf)

    results_clf.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test_clf, y_pred_clf), 4),
        'Precision': round(precision_score(y_test_clf, y_pred_clf, average='weighted'), 4),
        'Recall':    round(recall_score(y_test_clf, y_pred_clf, average='weighted'), 4),
        'F1-score':  round(f1_score(y_test_clf, y_pred_clf, average='weighted'), 4),
    })

results_clf_df = pd.DataFrame(results_clf)
print(results_clf_df.to_string(index=False))


###Learning curves para clasificación

In [ ]:
def plot_learning_curve_clf(model, X, y, preprocessor, title):
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    train_sizes, train_scores, test_scores = learning_curve(
        pipeline, X, y,
        cv=5,
        scoring='accuracy',
        train_sizes=np.linspace(0.1, 1.0, 8),
        n_jobs=-1
    )

    train_mean = train_scores.mean(axis=1)
    test_mean  = test_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    test_std   = test_scores.std(axis=1)

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, marker='o', label='Train Accuracy')
    plt.plot(train_sizes, test_mean,  marker='o', label='Test Accuracy')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)
    plt.fill_between(train_sizes, test_mean  - test_std,  test_mean  + test_std,  alpha=0.15)
    plt.title(title)
    plt.xlabel('Training Size')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

for name, model in models_clf.items():
    plot_learning_curve_clf(model, X_clf, y_clf, preprocessor_clf,
                             f'Learning Curve - {name} (Classification)')


# Exercise 2 - Boosting Methods

In this exercise, three boosting methods were implemented for both regression and classification tasks: AdaBoost, Gradient Boosting, and XGBoost.

For the regression task, the **meta (ID: 566)** dataset from OpenML was used.  
For the classification task, the **kr-vs-kp (ID: 3)** dataset from OpenML was used.

A preprocessing pipeline was applied to handle missing values and encode categorical variables, ensuring compatibility with the implemented models. The performance of each method was evaluated through appropriate metrics and learning curves using:

`train_sizes = np.linspace(0.1, 1.0, 8)`

The objective of this exercise is to compare the predictive behavior, generalization capacity, and robustness of boosting-based ensemble methods.

## Analysis of Boosting Methods for Regression

Three boosting regressors were evaluated on the **meta (ID: 566)** dataset. The preprocessing pipeline applies median imputation for numerical features and one-hot encoding for categorical features (`DS_Name`, `Alg_Name`), which encode algorithm-dataset context and carry useful predictive information.

**AdaBoost** (R2: 0.0262, RMSE: 146.80) uses `max_depth=3` trees with `learning_rate=0.5` and `n_estimators=200`. Reducing the learning rate from the default (1.0) while increasing the number of estimators improved stability on this noisy dataset, raising R2 from -0.53 to positive.

**Gradient Boosting** (R2: 0.0893, RMSE: 141.96) achieves the best result among the three methods with `learning_rate=0.05`, `max_depth=4`, and `subsample=0.8`. The low learning rate with row subsampling provides the best bias-variance balance on this dataset.

**XGBoost** (R2: ~0.06) uses its default hyperparameters (`learning_rate=0.3`, `max_depth=6`), which proved more effective than custom tuning on this small, noisy dataset. Aggressive customization (lower learning rate, shallower trees) reduced performance, suggesting that the default configuration is already well-suited for this feature space.

All R2 values are close to zero, which is expected for the meta dataset: it records algorithm performance across heterogeneous benchmark datasets, making the target variable inherently noisy and difficult to predict. This is a known property of meta-learning benchmarks. The learning curves illustrate how each model balances training fit and generalization as the training size increases.

## Analysis of Boosting Methods for Classification

All three boosting classifiers achieved strong performance on the **kr-vs-kp (ID: 3)** dataset, with accuracy values ranging from 96.25% to 97.34%.

**AdaBoost** (Accuracy: 96.25%, F1: 0.9625) uses decision trees with `max_depth=2` as weak learners and a `learning_rate=0.5`, which moderates the contribution of each estimator. It builds the ensemble sequentially by increasing the weight of misclassified samples, forcing each new tree to focus on the hardest examples.

**Gradient Boosting** (Accuracy: 97.34%, F1: 0.9734) optimizes the log-loss sequentially using residual corrections. The combination of `learning_rate=0.05`, `max_depth=4`, and `subsample=0.8` (stochastic gradient boosting) provides strong generalization by preventing individual trees from overfitting to the training set.

**XGBoost** (Accuracy: 97.34%, F1: 0.9734) matches Gradient Boosting's performance thanks to its built-in regularization via `colsample_bytree=0.8` (feature subsampling per tree). Both methods reach the same accuracy, confirming that the dataset's patterns are well-captured at this level of complexity.

The `stratify=y` option in the train/test split ensures class balance in both sets, yielding more reliable metric estimates. The learning curves with confidence bands show stable convergence across all three methods, with training and test accuracy converging as sample size grows, confirming low variance and good generalization.

## Comparative Discussion

Bagging and Boosting are both ensemble learning strategies, but they differ in the way they construct their models.

Bagging reduces variance by training multiple models independently on bootstrap samples and averaging their predictions. This makes it useful for improving stability and reducing overfitting.

Boosting, in contrast, builds models sequentially. Each new estimator focuses on correcting the errors made by the previous ones. Because of this, boosting usually reduces both bias and variance, often achieving better predictive performance than bagging.

Among the boosting methods implemented in this exercise, XGBoost is expected to provide the strongest results due to its efficient optimization process and built-in regularization. Gradient Boosting also tends to perform well, while AdaBoost may be more sensitive depending on the characteristics of the dataset.

Therefore, boosting methods can be considered more powerful for complex learning tasks, although they are also computationally more demanding and require more careful tuning.